# Datamine PLOTFR Process Example

This notebook demonstrates how to configure and run the **`plotfr`** process wrapper in `dmstudio`.

## Process Description

## PLOTFR

Generates a frame plot, consisting of a frame, a grid or tick marks, with annotation on each axis, and user defined titles on each axis.

This is a general process, which may be used to add a frame and axes to any plot. The usual method of operation is to generate a frame plot when the user is satisfied with the plot of the data to which the frame is to be added. The components are:-

##### Frame

A box around the data area (the X and Y axes) and an optional box around the full plot area (negative values of @**IGRID** parameter). The data and picture area sizes are taken from the prototype, or defined to be the screen size if the null plot prototype plotprot is used.

##### Numbering

Annotation at each tick mark or each grid mark.

##### Tick Marks

* **Out** (Tickmarks outwards from the box around the data area.):

* **In** (Tickmarks inwards from the box around the data area.):

##### Grid

A full grid of intersecting lines in place of the tickmarks.

##### Interior Crosses

A set of interior crosses marking the intersection points of the grid lines.

In addition, up to 80 characters of label are permitted on each axis. These are prompted for. The labels start from the origin (bottom left-hand corner of the data area), so that spaces should be inserted at the start of the label to space the text along the axes.

Normally tickmarks and grids start from the **XMIN, YMIN** values; however these may optionally be started at any required position by use of the optional parameters @**XGSTART** , @**YGSTART**.

If a number of plots are to be generated with the same scales and axes, then it is only necessary to generate a single frame using **PLOTFR**. This can then be joined to any data plot required by using the APPEND process.

### Input Files:

* **proto** (Plot Prototype File):
  Plot prototype file. Must contain the fields **X, Y, S1, S2** and **CODE** (numeric,
  explicit) and **XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE** (numeric, implicit). If these
  last 6 values set in **PROTO** , then corresponding parameters need not be set.
  Required=Yes

### Output Files:

* **plot** (Plot):
  Output plot file.
  Required=Yes

### Fields:

### Parameters:

* **xinc**:
  Grid increment on X axis.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **yinc**:
  Grid increment on Y axis.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **ndx**:
  Annotation decimal places on X axis.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **ndy**:
  Annotation decimal places on Y axis.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **igrid**:

* **Options** (0: frame only;; 1: frame + outwards ticks;; 2: frame + crosses at grid):
  intersections;; 3: frame + inwards ticks;; 4: grid;; 5-9: as 0-4 minus frame.; 10: as 4
  but anno- tation parallel to grid lines.; 11-2: 0 as 1-10 with annotation on right and
  top as well. Negative values of IGRID give an additional frame around the full plot
  area.
  Range=0,20
  Values=Undefined
  Default=0
  Required=Yes

* **noxaxis**:
  Suppresses plotting of X-axis.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **noyaxis**:
  Suppresses plotting of Y-axis.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xgstart**:
  Start point of X grid, ticks Default is **XMIN**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ygstart**:
  Start point of Y grid, ticks Default is **YMIN**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **charsize**:
  Character size in millimetres (3).
  Range=Undefined
  Values=Undefined
  Default=3
  Required=No

* **aspratio**:
  Aspect ratio, width / ht. for chars (0.9).
  Range=Undefined
  Values=Undefined
  Default=0.9
  Required=No

* **append**:
  Plot append flag. If set to 1 then the new plot will be appended to the PLOT file, if
  it exists and is a valid plot file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **xmin**:
  Minimum value of X for plot. None of **XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE** need be
  set if this information is already in the prototype.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xmax**:
  Maximum value of X for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymin**:
  Minimum value of Y for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymax**:
  Maximum value of Y for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xscale**:
  X scale in user data units per millimetre.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **yscale**:
  Y scale in user data units per millimetre.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('plotfr')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute plotfr
print("Running plotfr...")
dm_cmd.plotfr(
    proto_i='t_mod1',  # required
    plot_o='t_plotfr_out',  # required
    xinc_p='required_val',  # required
    yinc_p='required_val',  # required
    ndx_p='required_val',  # required
    ndy_p='required_val',  # required
    igrid_p='required_val',  # required
    # noxaxis_p='optional',  # optional
    # noyaxis_p='optional',  # optional
    # xgstart_p='optional',  # optional
    # ygstart_p='optional',  # optional
    # charsize_p=3,  # optional
    # aspratio_p=0.9,  # optional
    # append_p=0,  # optional
    # xmin_p='optional',  # optional
    # xmax_p='optional',  # optional
    # ymin_p='optional',  # optional
    # ymax_p='optional',  # optional
    # xscale_p='optional',  # optional
    # yscale_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("plotfr execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_plotfr_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")